# Гибридный поиск кандидатов (Avito)

Импорт необходимых библиотек

In [ ]:
import os
import re
import gc
import time
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

## Настройки

Пути к датасэтам, а также веса модели полученные экспериментально 

In [ ]:
TRAIN_PATH = "../data/train.parquet"
BENCH_QUERIES_PATH = "../data/benchmark_queries.parquet"
BENCH_ITEMS_PATH = "../data/benchmark_items.parquet"
OUTPUT_PATH = "../final_output/answer.csv"
TOP_K = 50

LOC_MODE = "mul"          
LOC_W = 8.0               
POP_W = 0.2               
DENSE_W = 0.6             
CAT_W = 0.15              
DELIV_LOC_SCALE = 1.0    
DENSE_SPREAD = 0.15       

USE_FILTERS = True        # текст фильтров в запрос BM25
DENSE_USE_FILTERS = True  # текст фильтров в запрос для эмбеддинга
LEMMATIZE = True

MODEL_NAME = "intfloat/multilingual-e5-small"  
MAX_SEQ_LEN = 64
CACHE_DIR = "emb_cache"
ENCODE_BATCH = 256
CHUNK = 50_000
IS_E5 = "e5" in MODEL_NAME.lower()
Q_PREFIX = "query: " if IS_E5 else ""
D_PREFIX = "passage: " if IS_E5 else ""


QUICK_TEST = False       # Прогон, чтобы проверить, что код не падает
QUICK_TRAIN_ROWS = 30_000
QUICK_BENCH_ITEMS = 10_000
QUICK_VAL_QUERIES = 200
FULL_VAL_QUERIES = 1000

TRAIN_COLS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
    "item_id", "item_title_raw", "item_description_raw",
    "item_infm_params_text", "item_category_id", "item_location_id",
    "item_rating_reviews_count",
]
ITEM_COLS = [
    "item_id", "item_title_raw", "item_description_raw",
    "item_infm_params_text", "item_category_id", "item_location_id",
    "item_rating_reviews_count",
]

## Предобработка текста

**Токенизация:** нижний регистр, `[a-zа-яё0-9]+` (русские и латинские буквы + цифры).

**Лемматизация (`pymorphy3`):** приводит слово к начальной форме («бани», «баню» → «баня»). Кэш через `@lru_cache` — каждое уникальное слово разбирается один раз.

**`build_item_text`:** заголовок дублируется (×2) — в заголовке сконцентрирован смысл. Описание обрезается до 300 символов, чтобы не размывать индекс редким мусором.

**`build_query_text`:** текст запроса + фильтры (если `use_filters=True`).

In [ ]:
TOKEN_RE = re.compile(r"[a-zа-яё0-9]+", re.IGNORECASE)
_CYR = re.compile(r"[а-яё]+")
try:
    import pymorphy3
    _MORPH = pymorphy3.MorphAnalyzer()
except ImportError:
    _MORPH = None


@lru_cache(maxsize=None)
def lemma(word):
    if not _CYR.fullmatch(word):
        return word
    if _MORPH is None:
        raise ImportError("Для лемматизации нужен pymorphy3: pip install pymorphy3")
    return _MORPH.parse(word)[0].normal_form


def tokenize(text):
    """Строка → список токенов."""
    if not isinstance(text, str) or not text:
        return []
    words = TOKEN_RE.findall(text.lower())
    return [lemma(w) for w in words] if LEMMATIZE else words


def build_item_text(title, params, desc):
    """Документ объявления: Заголовок ×2 + параметры + описание[:300]."""
    title, params, desc = str(title or ""), str(params or ""), str(desc or "")[:300]
    return f"{title} {title} {params} {desc}"


def build_query_text(row, use_filters=True):
    """Текст запроса: сам запрос + фильтры."""
    q = str(row.get("search_query", "") or "")
    if not use_filters:
        return q
    return f"{q} {row.get('search_infm_params_text', '') or ''}"

## BM25 на разреженной матрице

Стандартный BM25 (`k1=1.5, b=0.75`), но вклад каждого слова в каждый документ считается **один раз при построении индекса**. Тогда скор запроса — сумма нескольких столбцов (~1 мс), а не питоновский цикл по 500k документов на каждое слово (~1 сек).


In [ ]:
class SparseBM25:
    """BM25 на разреженной матрице."""

    def __init__(self, tokenized_corpus, k1=1.5, b=0.75, epsilon=0.25):
        self.vec = CountVectorizer(analyzer=lambda toks: toks)
        tf = self.vec.fit_transform(tokenized_corpus).tocsr().astype(np.float32)
        n_docs = tf.shape[0]
        df = np.diff(tf.tocsc().indptr)
        idf = np.log(n_docs - df + 0.5) - np.log(df + 0.5)
        idf = np.where(idf < 0, epsilon * idf.mean(), idf).astype(np.float32)
        dl = np.asarray(tf.sum(axis=1)).ravel()
        norm = k1 * (1 - b + b * dl / dl.mean())
        rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
        tfv = tf.data
        tf.data = idf[tf.indices] * tfv * (k1 + 1) / (tfv + norm[rows])
        self.W = tf.tocsc()
        self.vocab = self.vec.vocabulary_

    def get_scores(self, query_tokens):
        counts = {}
        for t in query_tokens:
            j = self.vocab.get(t)
            if j is not None:
                counts[j] = counts.get(j, 0) + 1
        if not counts:
            return np.zeros(self.W.shape[0], dtype=np.float32)
        cols = np.fromiter(counts.keys(), dtype=np.int64)
        w = np.fromiter(counts.values(), dtype=np.float32)
        return np.asarray(self.W[:, cols] @ w).ravel()

## Эмбеддинги с кэшем

Используется `intfloat/multilingual-e5-small` — компактная модель, 
работает локально. Для E5-моделей нужны префиксы `query:` для запросов 
и `passage:` для документов — без них качество падает.

Эмбеддинги сохраняются в `emb_cache/` по `item_id` в float16 — при 
повторных запусках уже закодированные объявления не пересчитываются. 
Сохранение идёт чанками по 50k, чтобы прерванный прогон можно было 
продолжить с места остановки.

Если папки `emb_cache/` нет — она создаётся автоматически при первом 
запуске. Первое кодирование занимает 1.5–2 часа на CPU или 1–2 минуты 
на GPU (Colab T4).

In [ ]:
_MODEL = None


def get_model():
    global _MODEL
    if _MODEL is None:
        from sentence_transformers import SentenceTransformer
        _MODEL = SentenceTransformer(MODEL_NAME)
        _MODEL.max_seq_length = MAX_SEQ_LEN
    return _MODEL


def _cache_paths():
    """Пути к файлам кэша. Имя зависит от MODEL_NAME."""
    slug = MODEL_NAME.replace("/", "__")
    os.makedirs(CACHE_DIR, exist_ok=True)
    return (os.path.join(CACHE_DIR, f"{slug}_ids.npy"),
            os.path.join(CACHE_DIR, f"{slug}_emb.npy"))


def get_embeddings(items_df):
    """
    Матрица эмбеддингов, выровненная с items_df.
    Недостающие item_id кодируются и дописываются в кэш.
    """
    ids_p, emb_p = _cache_paths()
    if os.path.exists(ids_p) and os.path.exists(emb_p):
        cached_ids, cached = np.load(ids_p), np.load(emb_p)
    else:
        cached_ids, cached = np.array([], dtype="U16"), None

    known = set(cached_ids.tolist())
    item_ids = items_df["item_id"].astype(str).to_numpy()
    missing = np.array([k for k, i in enumerate(item_ids) if i not in known], dtype=np.int64)
    print(f"  [эмбеддинги] в корпусе {len(item_ids)}, в кэше {len(item_ids) - len(missing)}, "
          f"кодируем {len(missing)}")

    if len(missing):
        model = get_model()
        titles = items_df["item_title_raw"].astype(str).to_numpy()
        params = items_df["item_infm_params_text"].astype(str).to_numpy()
        for start in range(0, len(missing), CHUNK):
            part = missing[start:start + CHUNK]
            docs = [f"{D_PREFIX}{titles[k]}. {params[k]}" for k in part]
            t0 = time.time()
            emb = model.encode(docs, batch_size=ENCODE_BATCH, normalize_embeddings=True,
                               show_progress_bar=True).astype(np.float16)
            cached = emb if cached is None else np.concatenate([cached, emb])
            cached_ids = np.concatenate([cached_ids, item_ids[part].astype("U16")])
            np.save(emb_p, cached)
            np.save(ids_p, cached_ids)
            print(f"  [эмбеддинги] {min(start + CHUNK, len(missing))}/{len(missing)} "
                  f"({time.time() - t0:.0f} сек)")

    pos = {i: k for k, i in enumerate(cached_ids.tolist())}
    rows = np.fromiter((pos[i] for i in item_ids), dtype=np.int64, count=len(item_ids))
    return cached[rows].astype(np.float32)

## Класс `HybridGenerator`

Строит все сигналы один раз при инициализации:

1. **BM25-индекс** — по лемматизированному тексту объявлений.
2. **Эмбеддинги** — из кэша или кодирует недостающие.
3. **Локации** — массив `item_location_id`.
4. **Категории** — массив `item_category_id` (используется как признак, но `CAT_W=0`).
5. **Популярность** — `log1p(reviews) / max(log1p(reviews))`.

**`parts(row)`** — считает BM25 один раз, остальные сигналы берёт из 
подготовленных массивов. Разложение на компоненты нужно, чтобы перебирать 
веса в экспериментах без пересчёта BM25.

**`combine(p, ...)`** — итоговая формула. В режиме `"mul"` локация входит 
множителем: при совпадении растёт весь скор, а не только слагаемое за локацию.

**`top_index`** — через `argpartition`, O(N) вместо полной сортировки корпуса.

In [ ]:
class HybridGenerator:
    """
    score = (BM25_norm + dense_w*dense + cat_w*cat) * (1 + loc*loc_w_eff) + pop*pop_w
    loc_w_eff = loc_w * deliv_scale для запросов с доставкой.
    """

    def __init__(self, items_df, dense_w=DENSE_W, cat_w=CAT_W, loc_w=LOC_W, pop_w=POP_W,
                 deliv_scale=DELIV_LOC_SCALE, loc_mode=LOC_MODE, spread=DENSE_SPREAD,
                 use_filters=USE_FILTERS, dense_filters=DENSE_USE_FILTERS):
        self.items_df = items_df.reset_index(drop=True)
        self.dense_w, self.cat_w, self.loc_w, self.pop_w = dense_w, cat_w, loc_w, pop_w
        self.deliv_scale, self.loc_mode, self.spread = deliv_scale, loc_mode, spread
        self.use_filters, self.dense_filters = use_filters, dense_filters
        df = self.items_df
        self.item_ids = df["item_id"].values

        # BM25-индекс
        t0 = time.time()
        texts = [build_item_text(t, p, d) for t, p, d in
                 zip(df["item_title_raw"], df["item_infm_params_text"], df["item_description_raw"])]
        corpus = [tokenize(t) for t in texts]
        print(f"  [время] токенизация {len(df)} объявлений: {time.time() - t0:.1f} сек")
        t0 = time.time()
        self.bm25 = SparseBM25(corpus)
        print(f"  [время] BM25-индекс: {time.time() - t0:.1f} сек")

        # Доп. сигналы
        self.item_locations = df["item_location_id"].values
        self.item_cats = df["item_category_id"].values
        rc = df["item_rating_reviews_count"].fillna(0).clip(lower=0).to_numpy(dtype=np.float64)
        lg = np.log1p(rc)
        self.item_pop = (lg / max(lg.max(), 1e-9)).astype(np.float32)

        # Эмбеддинги (из кэша или кодируются)
        self.emb = get_embeddings(df)
        self._qcache = {}

    # ---- запросы: эмбеддинги ----
    def _qtext(self, row):
        return Q_PREFIX + build_query_text(row, self.dense_filters).strip()

    def prefetch_queries(self, queries_df):
        """Заранее кодирует эмбеддинги всех уникальных запросов — быстрее, чем по одному."""
        texts = list(dict.fromkeys(self._qtext(r) for _, r in queries_df.iterrows()))
        texts = [t for t in texts if t not in self._qcache]
        if texts:
            embs = get_model().encode(texts, batch_size=128, normalize_embeddings=True,
                                      show_progress_bar=True)
            self._qcache.update(zip(texts, embs.astype(np.float32)))

    def _qemb(self, row):
        t = self._qtext(row)
        if t not in self._qcache:
            self._qcache[t] = get_model().encode(t, normalize_embeddings=True).astype(np.float32)
        return self._qcache[t]

    def parts(self, row):
         """Возвращает компоненты скора для одного запроса."""
        n = len(self.items_df)
        zeros = np.zeros(n, np.float32)

        # BM25
        tokens = tokenize(build_query_text(row, self.use_filters))
        text = zeros
        if tokens:
            text = self.bm25.get_scores(tokens)
            top = text.max()
            if top > 0:
                text = text / top

        cos = self.emb @ self._qemb(row)
        dense = np.clip(1.0 - (cos.max() - cos) / self.spread, 0.0, 1.0).astype(np.float32)

        sl, sc, d = row.get("search_location_id"), row.get("search_category"), row.get("search_is_delivery_search")
        loc = (self.item_locations == sl).astype(np.float32) if pd.notna(sl) else zeros
        cat = (self.item_cats == sc).astype(np.float32) if pd.notna(sc) else zeros
        deliv = bool(pd.notna(d) and int(d) == 1)

        return dict(text=text, dense=dense, cat=cat, loc=loc, pop=self.item_pop, deliv=deliv)

    def combine(self, p, dense_w=None, cat_w=None, loc_w=None, pop_w=None, deliv_scale=None):
        dense_w = self.dense_w if dense_w is None else dense_w
        cat_w = self.cat_w if cat_w is None else cat_w
        loc_w = self.loc_w if loc_w is None else loc_w
        pop_w = self.pop_w if pop_w is None else pop_w
        deliv_scale = self.deliv_scale if deliv_scale is None else deliv_scale

        base = p["text"] + dense_w * p["dense"] + cat_w * p["cat"]
        lw = loc_w * (deliv_scale if p["deliv"] else 1.0)

        if self.loc_mode == "mul":
            return base * (1.0 + p["loc"] * lw) + p["pop"] * pop_w
        return base + p["loc"] * lw + p["pop"] * pop_w

    @staticmethod
    def top_index(scores, top_k=TOP_K):
        top_k = min(top_k, len(scores))
        idx = np.argpartition(scores, -top_k)[-top_k:]
        return idx[np.argsort(scores[idx])[::-1]]

    def search(self, row, top_k=TOP_K):
        return list(self.item_ids[self.top_index(self.combine(self.parts(row)), top_k)])

## Подготовка валидации

`prepare_validation`:

1. Читает train (при `QUICK_TEST=True` — только 30k строк).
2. Строит `_qk` — ключ запроса (query + location + delivery + filters + category).
3. Откладывает 15% уникальных ключей (но не больше 1000) в валидацию.
4. Релевантные для запроса — все item_id, показанные по этому ключу в train.

**Оптимизации памяти:** `del train; gc.collect()` после сборки пула; `desc` обрезается до 300 символов сразу; склейка через `~isin` вместо `concat().drop_duplicates()`.

In [56]:
def load_train(n_rows=None):
    """Читает train.parquet (только TRAIN_COLS). n_rows — первые n строк."""
    if n_rows is None:
        return pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLS)
    import pyarrow.parquet as pq
    batch = next(pq.ParquetFile(TRAIN_PATH).iter_batches(batch_size=n_rows, columns=TRAIN_COLS))
    return batch.to_pandas()


def prepare_validation(bench_items_df=None):
    """(items_pool, query_features, relevant) для локальных экспериментов."""
    train = load_train(QUICK_TRAIN_ROWS if QUICK_TEST else None)
    print(f"  train: {len(train)} строк{'  [БЫСТРЫЙ РЕЖИМ]' if QUICK_TEST else ''}")

    train["_qk"] = (train["search_query"].astype(str) + "|" +
                    train["search_location_id"].astype(str) + "|" +
                    train["search_is_delivery_search"].astype(str) + "|" +
                    train["search_infm_params_text"].astype(str) + "|" +
                    train["search_category"].astype(str))

    keys = np.array(train["_qk"].unique(), dtype=object)
    np.random.default_rng(42).shuffle(keys)
    cap = QUICK_VAL_QUERIES if QUICK_TEST else FULL_VAL_QUERIES
    val_keys = set(keys[:min(max(1, int(0.15 * len(keys))), cap)])

    val_rows = train[train["_qk"].isin(val_keys)]
    relevant = val_rows.groupby("_qk")["item_id"].apply(set).to_dict()
    query_features = val_rows.drop_duplicates("_qk").set_index("_qk")

    pool_train = train.drop_duplicates("item_id")[ITEM_COLS].copy()
    pool_train["item_description_raw"] = pool_train["item_description_raw"].astype(str).str.slice(0, 300)
    del train, val_rows
    gc.collect()

    if bench_items_df is None:
        bench_items_df = pd.read_parquet(BENCH_ITEMS_PATH, columns=ITEM_COLS)
    if QUICK_TEST and QUICK_BENCH_ITEMS < len(bench_items_df):
        bench_items_df = bench_items_df.sample(QUICK_BENCH_ITEMS, random_state=42)
    pad = bench_items_df[ITEM_COLS].copy()
    pad["item_description_raw"] = pad["item_description_raw"].astype(str).str.slice(0, 300)
    pad = pad[~pad["item_id"].isin(set(pool_train["item_id"]))]
    pool = pd.concat([pool_train, pad], ignore_index=True)
    print(f"Корпус для валидации: {len(pool)} (train: {len(pool_train)}, из benchmark: {len(pad)})")
    return pool, query_features, relevant

## Метрики и перебор весов

**`eval_configs`:** для каждого запроса `parts()` считается один раз, потом перебираются конфиги весов — просто прибавляя `dense_w * dense`, `cat_w * cat` и т.д. Это в разы быстрее, чем строить `HybridGenerator` заново на каждый конфиг.

In [57]:
def eval_configs(gen, queries, relevant, cfgs):
    """Recall@50 для списка конфигов весов; parts на запрос считаются один раз."""
    rec = [[] for _ in cfgs]
    for qkey, qrow in queries.iterrows():
        rel = relevant[qkey]
        p = gen.parts(qrow)
        for i, c in enumerate(cfgs):
            top = gen.top_index(gen.combine(p, **c))
            rec[i].append(len(set(gen.item_ids[top]) & rel) / len(rel))
    return [float(np.mean(r)) for r in rec]

## `run_experiments` — подбор весов на валидации

После — переносим найденные значения в **блок настроек** 

In [58]:
def run_experiments(bench_items_df=None, n_queries=1000,
                    dense_grid=(0, 0.3, 0.6, 1.0, 1.5), cat_grid=(0, 0.1, 0.2, 0.4),
                    loc_grid=(2, 4, 8), deliv_grid=(1.0, 0.5, 0.25)):
    """Подбор весов на локальной валидации."""
    pool, qf, relevant = prepare_validation(bench_items_df)
    queries = qf.iloc[:n_queries]
    gen = HybridGenerator(pool)
    gen.prefetch_queries(queries)

    # Этап 1: dense_w × cat_w
    cfgs = [dict(dense_w=d, cat_w=c) for d in dense_grid for c in cat_grid]
    t0 = time.time()
    res = eval_configs(gen, queries, relevant, cfgs)
    table = pd.DataFrame(
        np.array(res).reshape(len(dense_grid), len(cat_grid)),
        index=[f"dense={d}" for d in dense_grid],
        columns=[f"cat={c}" for c in cat_grid],
    )
    print(f"\n--- Recall@{TOP_K}: dense_w × cat_w ({time.time() - t0:.0f} сек) ---")
    print(table.round(4).to_string())
    print("Ячейка dense=0, cat=0 — это BM25 без гибрида (референс).")
    best1 = cfgs[int(np.argmax(res))]
    print(f">>> этап 1: {best1}  Recall={max(res):.4f}")

    # Этап 2: loc_w × ослабление локации для доставки
    cfgs2 = [dict(best1, loc_w=l, deliv_scale=s) for l in loc_grid for s in deliv_grid]
    res2 = eval_configs(gen, queries, relevant, cfgs2)
    table2 = pd.DataFrame(
        np.array(res2).reshape(len(loc_grid), len(deliv_grid)),
        index=[f"loc_w={l}" for l in loc_grid],
        columns=[f"deliv×{s}" for s in deliv_grid],
    )
    print(f"\n--- Recall@{TOP_K}: loc_w × ослабление для доставки ---")
    print(table2.round(4).to_string())
    bj = int(np.argmax(res2))
    print(f">>> этап 2: {cfgs2[bj]}  Recall={res2[bj]:.4f}")
    print("\nПеренеси в НАСТРОЙКИ: DENSE_W, CAT_W, LOC_W, DELIV_LOC_SCALE. "
          "Если лучшее на краю сетки — расширь её.")

## Диагностика: категория и доставка

`run_diag` показывает на полном train:

- **доля совпадений категории** запроса и объявления;
- **разбивка по флагу доставки** — доли совпадений категории и локации, число строк.

**Что мы нашли:**

```
категория запроса == категории объявления: 100.0%

по флагу доставки:
                        cat_same  loc_same       n
search_is_delivery_search
0                            1.0     0.831  497661
1                            1.0     0.500      12
```

**Выводы:**

1. **Категория бесполезна** — в train 100% совпадений. В benchmark 2 категории на 99% item'ов. Поэтому `CAT_W = 0.0`.
2. **Запросов на доставку почти нет** (12 строк из 500k) — `DELIV_LOC_SCALE` нечем калибровать, оставлен 1.0.
3. **Локация сильно различается по флагу** (83% vs 50%) — логика ослабления локации при доставке верна, но в данных не проверяема.

In [59]:
def run_diag():
    """Сигнал категории и доставки в train."""
    tr = pd.read_parquet(TRAIN_PATH, columns=["search_category", "item_category_id",
                                              "search_is_delivery_search",
                                              "search_location_id", "item_location_id"])
    tr["cat_same"] = tr.search_category == tr.item_category_id
    tr["loc_same"] = tr.search_location_id == tr.item_location_id
    print(f"категория запроса == категории объявления: {tr.cat_same.mean():.1%}")
    g = tr.groupby("search_is_delivery_search")
    print("\nпо флагу доставки:")
    print(g[["cat_same", "loc_same"]].mean().join(g.size().rename("n")).round(3))

## Финальное предсказание и валидация `answer.csv`

`run_final` строит `answer.csv` для `benchmark_queries` по весам из **блока настроек**.

`validate_answer_csv` проверяет формат строго по требованиям задания:

- колонки ровно `['query_id', 'answer']`, без индекса;
- все `query_id` из `benchmark_queries` есть, лишних нет, без дублей;
- в каждой строке ≤ 50 `item_id`, без повторов;
- все `item_id` существуют в `benchmark_items`.

**Важно про `dtype=str`:** без него pandas может интерпретировать ID как числа и молча потерять ведущие нули/буквы.

In [60]:
def validate_answer_csv(bench_queries, bench_items):
    """Проверка формата answer.csv по требованиям задания."""
    df = pd.read_csv(OUTPUT_PATH, dtype=str)
    problems = []

    if list(df.columns) != ["query_id", "answer"]:
        problems.append(f"колонки: {list(df.columns)}")

    exp, got = set(bench_queries["query_id"].astype(str)), set(df["query_id"])
    if exp - got:
        problems.append(f"пропущено query_id: {len(exp - got)}")
    if got - exp:
        problems.append(f"лишних query_id: {len(got - exp)}")
    if df["query_id"].duplicated().any():
        problems.append("повторы query_id")

    valid = set(bench_items["item_id"].astype(str))
    for qid, ans in zip(df["query_id"], df["answer"].fillna("")):
        items = ans.split()
        if len(items) > TOP_K:
            problems.append(f"{qid}: больше {TOP_K} item_id")
        if len(items) != len(set(items)):
            problems.append(f"{qid}: повторы item_id")
        if any(i not in valid for i in items):
            problems.append(f"{qid}: несуществующие item_id")
        if len(problems) > 30:
            break

    print("Формат answer.csv в порядке." if not problems
          else "ПРОБЛЕМЫ:\n - " + "\n - ".join(problems[:30]))


def run_final():
    """Строит answer.csv для benchmark_queries по весам из НАСТРОЕК."""
    bench_items = pd.read_parquet(BENCH_ITEMS_PATH)
    bench_queries = pd.read_parquet(BENCH_QUERIES_PATH)
    gen = HybridGenerator(bench_items)
    gen.prefetch_queries(bench_queries)
    answers, t0 = [], time.time()
    for i, (_, qrow) in enumerate(bench_queries.iterrows(), 1):
        answers.append(" ".join(gen.search(qrow, TOP_K)))
        if i % 500 == 0:
            print(f"  {i}/{len(bench_queries)} ({time.time() - t0:.0f} сек)")
    pd.DataFrame({"query_id": bench_queries["query_id"].values, "answer": answers}) \
        .to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    print(f"Готово: {OUTPUT_PATH} ({len(answers)} строк)")
    validate_answer_csv(bench_queries, bench_items)

## Точка входа

Поставь флаги под свой сценарий:

- **`DO_DIAG = True`** — анализ данных (секунды).
- **`DO_EXPERIMENTS = False`** — подбор весов на валидации. **Поставь True только один раз**, потом верни False. По умолчанию отключено, чтобы `Run All` не занимал час.
- **`DO_FINAL = True`** — построить `answer.csv`. Это то, что отправляется на платформу.

**Если запускаешь через `Run All` первый раз** (кэша эмбеддингов нет) — `DO_EXPERIMENTS = False`, `DO_FINAL = False`, сначала дай `run_diag()` создать `HybridGenerator` в одной из своих ячеек и закэшировать эмбеддинги. Или поставь `DO_FINAL = True` — он сам закодирует и сохранит.

In [ ]:
if __name__ == "__main__" or True: 
    DO_DIAG = False
    DO_EXPERIMENTS = False            # True — подбор весов (занимает ~15 мин)
    DO_FINAL = True                   # True — построить answer.csv

    if DO_DIAG:
        run_diag()

    if DO_EXPERIMENTS:
        # Основная сетка
        run_experiments()
        # Расширенная сетка (по результатам первого прогона)
        run_experiments(
            n_queries=1000,
            dense_grid=(0.4, 0.6, 0.8),
            cat_grid=(0, 0.15),
            loc_grid=(0, 0.5, 1, 2, 4),
            deliv_grid=(1.0,),
        )

    if DO_FINAL:
        run_final()

  [время] токенизация 189212 объявлений: 33.9 сек
  [время] BM25-индекс: 9.0 сек
  [эмбеддинги] в корпусе 189212, в кэше 189212, кодируем 0


Batches: 100%|██████████| 20/20 [00:09<00:00,  2.18it/s]


  500/2452 (6 сек)
  1000/2452 (12 сек)
  1500/2452 (17 сек)
  2000/2452 (23 сек)
Готово: ../final_output/answer.csv (2452 строк)
Формат answer.csv в порядке.
